# Model Definition and Evaluation — Leafline (Tree-Crown Mapping, Kiel)

## Table of Contents
1. [Model Selection](#model-selection)
2. [Feature Engineering](#feature-engineering)
3. [Hyperparameter Tuning](#hyperparameter-tuning)
4. [Implementation](#implementation)
5. [Evaluation Metrics](#evaluation-metrics)
6. [Comparative Analysis](#comparative-analysis)

**Kurzfassung.** Wir finetunen ein vortrainiertes DeepTrees-Baumkronen-Modell auf Kieler
Luftbildern (RGBI + NDVI + Höhe/nDOM) und untersuchen isoliert die Effekte von Auflösung,
Jahreszeit, Eingangskanälen, Lernrate und Postprocessing. Bewertet wird **kronenweise
(Instanz-)F1** per IoU-Matching. Kernbefunde: getrennte Modelle je Auflösung, aber ein
Modell je Auflösung über beide Jahreszeiten; NDVI und nDOM helfen (v.a. Precision); der
**Recall ist die Decke** und wird von keinem der getesteten Hebel bewegt.


In [1]:
# Auswertung liest die committeten Ergebnis-CSVs (Training/Inferenz laufen via
# 3_Model/src/*.py unter dem nda-Konto, siehe 3_Model/README.md).
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
R = Path('runs')  # relativ zu 3_Model/

def micro(df, res=None):
    d = df if res is None else df[df.aufloesung == res]
    tp, fp, fn = d.tp.sum(), d.fp.sum(), d.fn.sum()
    p = tp/(tp+fp) if tp+fp else 0.0; r = tp/(tp+fn) if tp+fn else 0.0
    return round(2*p*r/(p+r) if p+r else 0.0, 3), round(p,3), round(r,3)

def load(p):
    p = Path(p); return pd.read_csv(p) if p.exists() else None

## Model Selection

Für die Instanz-Segmentierung einzelner Baumkronen aus hochauflösenden Luftbildern haben
wir drei Wege abgewogen:

1. **Ein eigenes Modell von Grund auf trainieren** — verworfen: erfordert sehr viele
   annotierte Kronen; unsere Ground-Truth-Menge ist klein (wenige Kieler Gebiete), und
   das Annotieren ist aufwändig und wurde extern erstellt.
2. **Ein generisches Foundation-Modell (SAM, „Segment Anything")** — verworfen als
   Kernmodell: SAM ist promptbasiert und nicht auf dichte, kleine Baumkronen aus
   Nadir-Luftbildern spezialisiert; es nutzt weder den NIR-Kanal noch Höhendaten und
   liefert keine kronentypischen Outline-/Distanz-Ausgaben für eine Watershed-Trennung
   aneinanderstoßender Kronen.
3. **Ein domänenspezifisches, vortrainiertes Baumkronen-Modell finetunen** — gewählt.

Wir wählen **DeepTrees / `freudenberg2022`** (U-Net, ResNet-Backbone). Es ist genau auf
Baumkronen-Delineation trainiert, gibt drei Karten aus (Kronenmaske, Kronen-Outline,
Distanztransformation), die per Watershed zu Einzelkronen werden, akzeptiert RGBI+NDVI und
lässt sich um einen Höhenkanal erweitern. Finetuning statt Neutraining passt zur
Datenknappheit und liefert zugleich einen **übertragbaren Workflow** (neue Stadt/Auflösung
mit wenig Daten adaptieren) — das eigentliche Projektziel.


## Feature Engineering

Aus den Rohdaten bauen wir pro Gebiet einen **6-Kanal-Stack** (`prepare_data.py`):

| # | Kanal | Herkunft |
|---|---|---|
| 0–3 | R, G, B, I (NIR) | DOP-Luftbild (÷255) |
| 4 | **NDVI** = (NIR−R)/(NIR+R) | berechnet, Vegetationsindex, auf [0,1] skaliert |
| 5 | **nDOM** (Höhe) | normalisiertes Oberflächenmodell, per-Kachel min-max |

Weitere Schritte über die Baseline hinaus:
- **Native Stacks je Auflösung/Jahreszeit** (7.5 cm Frühjahr, 20 cm Sommer/Frühjahr) ohne
  Umprojektion, damit die Bewertung je Auflösung fair und baseline-vergleichbar ist.
- **Ground Truth → 3 Bänder** (Maske | Outline | Distanztransformation), passend zu den
  Modellausgaben.
- **Konfigurierbare, jahreszeiten-bewusste Augmentierung** (`dataset.py`): geometrisch
  (Flips) + Helligkeit/Kontrast; die spektrale Sommer→Frühjahr-Simulation (NDVI/NIR/Rausch)
  ist einzeln zuschaltbar und für reine Frühjahrsläufe bewusst **aus** (Effekte isolieren).

Welche Kanäle real beitragen, quantifiziert die Kanal-Ablation bei 7.5 cm (unten): jeder
Kanal hilft — überwiegend über die Precision.


In [2]:
# Kanal-Ablation @7.5cm (alle mit getuntem PP 30/2), aus den eval_test.csv
rows=[]
for lbl, run in [('4ch RGBI','step1_rgbi_spring75'),('5ch +NDVI','step1_spring75'),
                 ('6ch +nDOM','step1_ndom_spring75')]:
    d = load(R/run/'eval_test.csv')
    if d is not None:
        f1,p,r = micro(d,'7.5cm')
        rows.append({'Kanäle':lbl,'F1':f1,'Precision':p,'Recall':r})
print(pd.DataFrame(rows).to_string(index=False) if rows else 'CSVs nicht gefunden')

   Kanäle    F1  Precision  Recall
 4ch RGBI 0.123      0.166   0.098
5ch +NDVI 0.150      0.218   0.115
6ch +nDOM 0.158      0.249   0.116


## Hyperparameter Tuning

Zwei Ebenen, jeweils datengetrieben:

**1. Lernrate — 3-fach Cross-Validation über ein LR-Raster {5e-5, 1e-4, 2e-4}**
(`train.py --cv-folds`). Ergebnis: das Optimum ist **auflösungsabhängig** (7.5 cm bevorzugt
5e-5, 20 cm 2e-4), aber die neu trainierten Best-LR-Modelle bringen **auf dem Test keinen
Gewinn** (Val steigt, Test nicht — die Val↔Test-Lücke). Wir behalten die feste **1e-4**.

**2. Postprocessing (Watershed) — pro Auflösung getunt** (`pp_sweep.py`, Sweep über
`min_dist × sigma`). Das PP ist auflösungsabhängig: 7.5 cm braucht **min_dist=30, sigma=2**
(viele Fragmente zusammenfassen), 20 cm **min_dist=10, sigma=3** (größeres min_dist
verschmilzt hier echte Nachbarkronen). Das PP hebt die **Precision**, nicht den Recall.


In [3]:
# CV-Ergebnis: mittleres Val-F1 je Lernrate (3 Folds)
for run in ['step1_spring75','step2_spring20']:
    d = load(R/run/'cv_results.csv')
    if d is not None:
        g = d.groupby('lr')['best_val_f1'].mean().round(3)
        print(run, '— mean val_F1 je LR:'); print(g.to_string()); print()

step1_spring75 — mean val_F1 je LR:
lr
0.00005    0.768
0.00010    0.751
0.00020    0.750

step2_spring20 — mean val_F1 je LR:
lr
0.00005    0.707
0.00010    0.719
0.00020    0.749



## Implementation

Finetuning in reinem PyTorch (`train.py`, kein Lightning):

- **Gewichtstransfer:** JIT-Checkpoint `freudenberg2022` (5-Kanal RGBI+NDVI) wird geladen;
  die erste Conv wird generisch an die Ziel-Kanalzahl angepasst (`model_utils.py`):
  Erweitern (5→6, neuer nDOM-Kanal = Mittel der vortrainierten Kanäle) oder Reduzieren
  (5→4, RGBI behalten). Übrige Schichten exakt übernommen.
- **Loss:** `mask + outline + 2·dist` — die Distanztransformation (Baumhöhe/-form,
  jahreszeiten-unabhängig) wird 2× gewichtet, um den Frühjahrs-Domänen-Gap abzufedern.
- **Optimierung:** Adam, Early Stopping (patience=10), auto-Resume aus `last.pt`.

**Experimentplan (`SCHEDULE.txt`), je ein Faktor isoliert:**

| Run | Daten | Kanäle | Zweck |
|---|---|---|---|
| step1 | 100% Frühjahr 7.5 cm | 5 | Basis-Finetune bei nativer Auflösung |
| step2 | 100% Frühjahr 20 cm | 5 | eigenes Modell für 20 cm |
| step3 | 50/50 Sommer+Frühjahr 20 cm | 5 | hält ein 20 cm-Modell beide Saisons? |
| step1_rgbi / step1_ndom | wie step1 | 4 / 6 | Kanal-Ablation (NDVI, nDOM) |

Die Datenzugriffe (`Data/Kiel`) laufen unter dem `nda`-Konto; Betriebsanleitung in
`3_Model/README.md`.


In [4]:
# Trainingsverlauf (pixelweises Val-F1) der Schedule-Modelle
fig, ax = plt.subplots(figsize=(9,4.5))
for run, c in [('step1_spring75','#2a78d6'),('step2_spring20','#008300'),('step3_mix20','#8a4fbe')]:
    d = load(R/run/'train_log.csv')
    if d is not None:
        ax.plot(d['epoch'], d['val_f1'], label=run, color=c, lw=2)
ax.set_xlabel('Epoche'); ax.set_ylabel('Val-F1 (pixelweise)'); ax.set_title('Trainingsverlauf')
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

## Evaluation Metrics

**Kronenweise (Instanz-)F1 per IoU-Matching** (IoU ≥ Schwelle zählt eine GT-Krone als
getroffen; greedy, jede GT höchstens einmal). Warum nicht pixelweise F1? Das pixelweise
Val-F1 misst „Baum/Nicht-Baum" je Pixel und weicht **stark** von der Einzelkronen-Leistung
ab (z.B. Val ~0.79, kronenweise Test ~0.16) — für das Ziel „einzelne Bäume zählen/vermessen"
ist die Instanz-Metrik die richtige. Sie ist zudem identisch zur Baseline-Auswertung.

Wir berichten **zwei IoU-Schwellen**: **0.5** (streng) und **0.3** — 0.3 ist für kleine
Baumkronen in Luftbildern üblicher, da eine kleine Krone bei 0.5 nahezu perfekte
Überlappung braucht. Aggregation als **Mikro-Schnitt** (TP/FP/FN über die Kacheln
summiert). Zur Fehleranalyse teilen wir GT-Kronen in **zero** (keine überlappende
Vorhersage), **partial** (überlappt, aber IoU<0.5) und **hit** ein.


In [5]:
# IoU-Schwellen-Kurve des besten 7.5cm-Modells (step1_ndom)
d = load(R/'step1_ndom_spring75/iou_curve_7.5cm.csv')
if d is not None:
    d = d.sort_values('iou_threshold')
    fig, ax = plt.subplots(figsize=(8,4.5))
    ax.plot(d.iou_threshold, d.f1, marker='o', label='F1'); ax.plot(d.iou_threshold, d.recall, marker='s', label='Recall')
    ax.axvline(0.5, color='gray', ls='--'); ax.axvline(0.3, color='green', ls='--')
    ax.set_xlabel('IoU-Schwelle'); ax.set_ylabel('Wert'); ax.set_title('step1_ndom @7.5cm: F1/Recall vs. IoU-Schwelle')
    ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()
    print(d[['iou_threshold','precision','recall','f1']].to_string(index=False))

 iou_threshold  precision  recall     f1
           0.1     0.8099  0.3784 0.5158
           0.2     0.7281  0.3402 0.4637
           0.3     0.5965  0.2787 0.3799
           0.4     0.4211  0.1967 0.2682
           0.5     0.2485  0.1161 0.1583


## Comparative Analysis

**Baseline vs. Finetuning** (kronenweise Mikro-F1). Baseline = un-finetuntes
`freudenberg2022` (PP 10/1); die Finetune-Modelle mit ihrem getunten Heim-PP:

| Auflösung | Baseline | Finetune (bestes Modell) |
|---|---|---|
| **7.5 cm** (Frühjahr) | **0.000** | **0.158** (step1_ndom, 6ch) — @IoU 0.3: **0.38** |
| **20 cm** (Sommer) | **0.340** | **0.317** (step3, 50/50) |
| **20 cm-spring** (Frühjahr) | **0.044** | **0.144** (step3, 50/50) |

**Verbesserungen:**
- **7.5 cm: 0.000 → 0.158.** Die Baseline (auf ~20 cm trainiert) erkennt bei nativer
  7.5 cm-Auflösung praktisch nichts (out-of-distribution); erst das Finetuning macht Kiels
  native Frühjahrsauflösung nutzbar.
- **20 cm-spring: 0.044 → 0.144 (~3.3×).**
- **Ein 20 cm-Modell reicht für beide Saisons:** step3 (50/50) hält Sommer (≈ Baseline)
  *und* verbessert Frühjahr sogar gegenüber dem reinen Frühjahrsmodell — ein separates
  Sommer-Modell ist nicht nötig.

**Gelernte Struktur:** getrennte Modelle je **Auflösung** (7.5 vs. 20 cm), ein Modell je
Auflösung über beide **Jahreszeiten**. Kanäle: RGBI 0.123 < +NDVI 0.150 < +nDOM 0.158
(alle Gewinne über die Precision).

**Rückschläge / Grenzen (Recall ist die Decke).** Bei IoU 0.5 liegt der Recall bei ~0.12
und wird von **keinem** Hebel bewegt — Postprocessing, Lernrate und Eingangskanäle ändern
ihn nicht (nur die Precision). Die Fehleranalyse (step1_ndom @7.5 cm) zeigt: ~40 % der
GT-Kronen werden gar nicht erkannt (*zero*), ~48 % sind lokalisiert, verfehlen aber die
0.5-Schwelle (*partial*) — visuell eine **Mischung aus Über-Segmentierung und Verschmelzen
benachbarter Kronen**. Bei IoU 0.3 verdoppelt sich die F1 (0.158 → 0.38), d.h. ein großer
Teil sind Near-Miss, keine Platzierungsfehler.

**Fazit & Ausblick.** Der Workflow adaptiert ein vortrainiertes Kronenmodell mit wenig
Daten erfolgreich auf eine neue Stadt/Auflösung (Baseline bei 7.5 cm unbrauchbar → nutzbar)
und zeigt, dass Auflösung und Jahreszeit die dominanten Faktoren sind. Der begrenzende
Faktor ist die **Erkennung/Trennung dicht stehender kleiner Kronen** — ein Daten-/
Annotations-Thema. Nächste Hebel (future work): den **Outline-Kanal stärken**
(PP `outline_multiplier`/`outline_exp` oder höheres Outline-Loss-Gewicht) gegen das
Verschmelzen, gezieltes Kleinkronen-Sampling und mehr/dichtere Annotationen.


In [6]:
# Vollständige Vergleichstabelle aus den CSVs (Heim-Auflösungen, getuntes PP)
tbl = {}
b = load(R/'baseline_eval.csv')
if b is not None:
    tbl['baseline'] = {res: micro(b,res)[0] for res in ['7.5cm','20cm','20cm-spring']}
for run in ['step1_spring75','step2_spring20','step3_mix20','step1_rgbi_spring75','step1_ndom_spring75']:
    d = load(R/run/'eval_test.csv')
    if d is not None:
        tbl[run] = {res: micro(d,res)[0] for res in d.aufloesung.unique()}
print(pd.DataFrame(tbl).T.to_string())

                     7.5cm   20cm  20cm-spring
baseline             0.000  0.340        0.044
step1_spring75       0.150    NaN          NaN
step2_spring20         NaN    NaN        0.113
step3_mix20            NaN  0.317        0.144
step1_rgbi_spring75  0.123    NaN          NaN
step1_ndom_spring75  0.158    NaN          NaN
